# Understanding the dataset

The dataset we will use is **UniRef50** and **InterPro**, where the UniRef is the clustering using UniProtKB protein sequenve, and InterPro is the annotation and it is linked to UniProtKB.

Therefore, we will use the cluster in UniRef50 -> get the UniProtKB id -> map to the InterPro annotation.

In [1]:
!zcat uniref50/uniref50.fasta.gz | head -n 2

>UniRef50_UPI002E2621C6 uncharacterized protein LOC134193701 n=1 Tax=Corticium candelabrum TaxID=121492 RepID=UPI002E2621C6
MGRIRVWVGTSIPNPVNAHQLVYLKGMAKTKKLILLLFVAAQPNFKEWSLDVDASTLVLT

gzip: stdout: Broken pipe


In [1]:
# save as make_ipr_vocab.py
import csv

entry_list = "InterPro/entry.list"
out_tsv = "InterPro/ipr_domain_entry.tsv"

domain_rows = []
with open(entry_list, "r") as f:
    # entry.list is typically tab-separated
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        # Common formats:
        # IPRxxxxxx  <type>  <name> ...
        ipr = parts[0]
        # Find a field that looks like a type/category
        # We'll scan for known category keywords.
        type_field = parts[1]
        if type_field == "Domain":
            name = parts[2] if len(parts) > 2 else ""
            domain_rows.append((ipr, name))

# Assign integer IDs (reserve 0 for NONE)
domain_rows.sort(key=lambda x: x[0])
with open(out_tsv, "w", newline="") as fo:
    w = csv.writer(fo, delimiter="\t")
    w.writerow(["ipr", "name", "idx"])
    for idx, (ipr, name) in enumerate(domain_rows, start=1):
        w.writerow([ipr, name, idx])

print(f"Wrote {len(domain_rows)} domain labels to {out_tsv}")


Wrote 17951 domain labels to InterPro/ipr_domain_entry.tsv


## Parse UniRef50 FASTA and store sequences keyed by RepID

In [ ]:
# save as build_uniref50_sqlite.py
import gzip
import sqlite3

FASTA = "uniref50/uniref50.fasta.gz"
DB = "uniref50/uniref50.sqlite"
LIMIT = None  # set e.g. 200000 for a pilot run

con = sqlite3.connect(DB)
cur = con.cursor()

cur.execute("PRAGMA journal_mode=WAL;")
cur.execute("PRAGMA synchronous=OFF;")
cur.execute("PRAGMA temp_store=MEMORY;")

cur.execute("""
CREATE TABLE IF NOT EXISTS protein (
  rep_id TEXT PRIMARY KEY,
  seq TEXT NOT NULL,
  len INTEGER NOT NULL
);
""")
con.commit()

def parse_rep_id(header: str) -> str:
    # Header example:
    # >UniRef50_UPI002E2621C6 uncharacterized ... RepID=UPI002E2621C6
    # We take token after >UniRef50_
    first = header.split()[0]
    assert first.startswith(">UniRef50_")
    return first[len(">UniRef50_"):]

batch = []
n = 0
with gzip.open(FASTA, "rt") as f:
    rep_id = None
    seq_chunks = []
    for line in f:
        line = line.rstrip("\n")
        if line.startswith(">"):
            if rep_id is not None:
                seq = "".join(seq_chunks)
                batch.append((rep_id, seq, len(seq)))
                n += 1
                if len(batch) >= 2000:
                    cur.executemany("INSERT OR REPLACE INTO protein(rep_id, seq, len) VALUES(?,?,?)", batch)
                    con.commit()
                    batch.clear()
                if LIMIT and n >= LIMIT:
                    break
            rep_id = parse_rep_id(line)
            seq_chunks = []
        else:
            seq_chunks.append(line.strip())

    if (not LIMIT or n < LIMIT) and rep_id is not None:
        seq = "".join(seq_chunks)
        batch.append((rep_id, seq, len(seq)))

if batch:
    cur.executemany("INSERT OR REPLACE INTO protein(rep_id, seq, len) VALUES(?,?,?)", batch)
    con.commit()

cur.execute("CREATE INDEX IF NOT EXISTS idx_protein_len ON protein(len);")
con.commit()
con.close()

print(f"Loaded {n if LIMIT else 'all'} proteins into {DB}")
